In [ ]:
import paho.mqtt.client as mqtt
import json
import uuid
import config # Imports your existing config.py

# 1. Load configuration dynamically
cfg = config.get_config()

# Define the wildcard topic for all VDA 5050 traffic for your fleet
# Format: interface/version/manufacturer/# (The '#' means "all sub-topics")
TOPIC = f"{cfg.mqtt_broker.vda_interface}/{cfg.vehicle.vda_version}/{cfg.vehicle.manufacturer}/#"

def on_connect(client, userdata, flags, reason_code, properties):
    print(f"Connected to broker at {cfg.mqtt_broker.host}:{cfg.mqtt_broker.port} (Code: {reason_code})")
    client.subscribe(TOPIC)
    print(f"Subscribed to wildcard topic: {TOPIC}")

def on_message(client, userdata, message):
    topic = message.topic
    try:
        # Parse the VDA 5050 JSON payload
        payload = json.loads(message.payload.decode('utf-8'))
        
        # Extract metadata from the topic path
        msg_type = topic.split('/')[-1] # e.g., 'state', 'visualization', 'order'
        serial = topic.split('/')[-2]   # e.g., 's0', 's1'
        
        # Print a clean summary instead of the entire JSON block
        print(f"[{msg_type.upper()}] Robot: {serial} | Time: {payload.get('timestamp')}")
        
        # NOTE: If you ever want to see the full payload, uncomment the line below:
        # print(json.dumps(payload, indent=2))
        
    except json.JSONDecodeError:
        print(f"Received non-JSON data on {topic}: {message.payload.decode()}")

# 2. Use a unique client ID so it doesn't conflict with main.py or VDAFleetManager
client_id = f"Jupyter_Monitor_{uuid.uuid4().hex[:8]}"
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=client_id)

client.on_connect = on_connect
client.on_message = on_message

# 3. Connect using your config.toml values
client.connect(cfg.mqtt_broker.host, int(cfg.mqtt_broker.port), 60)

# 4. Start the background thread
client.loop_start() 

print("VDA 5050 MQTT Monitor initialized and running in the background...")
